# Tier-0 · 0.1 — CTC baseline on AraMS-28k-HTR (crash-safe)

Two line-recognition baselines on the AraMS-28k-HTR **book-level** split:

1. **Kraken** — `ketos train` warm-started from the Muharaf recognition model
   (Zenodo `10.5281/zenodo.14295489`). The citable baseline.
2. **Own CRNN** — CNN → 2×BiLSTM(256) → CTC, height 64, AdamW 3e-4 cosine,
   batch 32, grad-clip 5.0, early stop on val-CER (patience 10). The one you
   ablate.

**Sanity anchors (RefLAM, Kraken on the AraMS-28k test split):** weighted CER
**23.31 %** — Maghrebi 32.71 / Naskh 22.62 / Ruq'ah 11.65. Wildly off ⇒ the
bug is in the crops or the charset, not the model.

**Crash-safe.** Everything durable lives on Google Drive; the CRNN checkpoints
every `SAVE_EVERY` steps *and* every epoch, Kraken every epoch (`--savefreq 1`).
Reopen + **Run all** after a disconnect and both resume from the last
checkpoint. Per-eval metrics are appended to `metrics.jsonl` so nothing is
lost even if the final cell never runs.

**Before running:** `Runtime → T4 GPU`; Colab secrets `HF_TOKEN` (dataset) and
`GH_TOKEN` (push the small results JSON). Run `07_colab_mirror_arams.ipynb`
first (or let cell 4 stream the mirror).

## 1. GPU

In [ ]:
import torch
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    print('GPU:', torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=memory.total,memory.used --format=csv
else:
    print('NO GPU — will run on CPU (much slower; big 4-bit models are skipped).')
    print('For a real run: Runtime -> Change runtime type -> T4 GPU.')

## 2. Drive — the durable store (checkpoints + resume live here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, json, time
WORK = pathlib.Path('/content/drive/MyDrive/arabic-ocr-bench/train/ctc-baseline')
(WORK / 'kraken').mkdir(parents=True, exist_ok=True)
(WORK / 'crnn').mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = '/content/drive/MyDrive/arabic-ocr-bench/hf_cache'
print('durable dir:', WORK)

## 3. Repo (for metrics/normalize) + auth + install

In [ ]:
# Bulletproof checkout: works whether the repo is missing, at the wrong
# path, or has local edits. Always ends up at the latest origin/main.
import os
REPO = '/content/arabic-ocr-benchmark'
# Copy Colab secrets into the environment. SWEEP_RUNNER_ID pins a STABLE
# name for this account (e.g. 'runner-a') so a restarted runner re-adopts
# its own in-flight batches instantly instead of waiting out the lease TTL.
for _k in ('GH_TOKEN', 'HF_TOKEN', 'SWEEP_RUNNER_ID'):
    try:
        from google.colab import userdata
        _v = userdata.get(_k)
        if _v: os.environ[_k] = _v
    except Exception:
        pass
token = os.environ.get('GH_TOKEN')
if not token:
    print('no GH_TOKEN secret — clone/push will be anonymous / read-only')
url = f"https://{(token + '@') if token else ''}github.com/rabeeh97/arabic-ocr-benchmark.git"
if os.path.isdir(REPO + '/.git'):
    os.system(f'git -C {REPO} fetch --depth 1 origin main && git -C {REPO} reset --hard origin/main')
    # drop untracked leftovers (e.g. a stale sweep_manifest.json) so the
    # manifest is always exactly what's in the repo. Keeps gitignored _cache.
    os.system(f'git -C {REPO} clean -fd -- results')
else:
    os.system(f'rm -rf {REPO} && git clone --depth 1 {url} {REPO}')
os.chdir(REPO)
# src-layout package: importable straight from the checkout even if the
# `pip install -e` cell fails or is skipped (this is what caused the old
# 'No module named arabic_ocr_bench' in the sweep cell).
import sys
for _p in (os.path.join(os.getcwd(), 'src'), os.getcwd()):
    if _p not in sys.path: sys.path.insert(0, _p)
print(os.popen('git log --oneline -3').read())

In [ ]:
import os, sys
for _p in (os.path.join(os.getcwd(), 'src'), os.getcwd()):
    if _p not in sys.path: sys.path.insert(0, _p)
from google.colab import userdata
for k in ('HF_TOKEN', 'GH_TOKEN'):
    try:
        v = userdata.get(k)
        if v: os.environ[k] = v
    except Exception as e:
        print('no', k, 'secret:', e)
if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
!pip install -q kraken
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 4. Materialise the data as an `images/` tree + manifests

Prefers the Kraken tar `07_colab_mirror_arams.ipynb` wrote to Drive; falls
back to streaming `<hf-user>/arams28k-htr-lines` and rebuilding the tree.

In [ ]:
import pathlib, tarfile, csv
DATA = pathlib.Path('/content/arams_htr')
TAR = pathlib.Path('/content/drive/MyDrive/arabic-ocr-bench/data/arams28k-htr-kraken.tar')
IMAGES = DATA / 'images'
if not IMAGES.exists():
    DATA.mkdir(parents=True, exist_ok=True)
    if TAR.exists():
        print('restoring from', TAR)
        with tarfile.open(TAR) as tf: tf.extractall(DATA)
    else:
        print('no Drive tar — rebuilding from the HF mirror')
        from datasets import load_dataset
        from huggingface_hub import whoami
        repo = f"{whoami()['name']}/arams28k-htr-lines"
        IMAGES.mkdir(parents=True, exist_ok=True)
        man = {s: open(DATA / f'{s}_manifest.txt', 'w') for s in ('train','val','test')}
        meta_rows = []
        for split in ('train', 'val', 'test'):
            for r in load_dataset(repo, split=split):
                sid = f"{r['book_id']}_{r['page_id']}_{r['line_idx']}"
                r['image'].save(IMAGES / f'{sid}.png')
                (IMAGES / f'{sid}.gt.txt').write_text(r['text'], encoding='utf-8')
                man[split].write(sid + '\n')
                meta_rows.append({'sample_id': sid, 'script': r.get('script',''),
                                  'zone': r.get('zone','main'), 'split': split})
        for f in man.values(): f.close()
        with open(DATA / 'metadata.csv', 'w', newline='') as fh:
            w = csv.DictWriter(fh, fieldnames=['sample_id','script','zone','split']); w.writeheader()
            w.writerows(meta_rows)
SPLIT = {s: [l.strip() for l in (DATA / f'{s}_manifest.txt').read_text().splitlines() if l.strip()]
         for s in ('train', 'val', 'test')}
SCRIPT = {}
with open(DATA / 'metadata.csv', newline='') as fh:
    for row in csv.DictReader(fh):
        k = next((row[c] for c in row if c.lower() in ('sample_id','id','filename','name')), None)
        if k: SCRIPT[pathlib.Path(k).stem.replace('.gt','')] = (row.get('script') or '').lower()
print({s: len(v) for s, v in SPLIT.items()}, '| scripts:', sorted(set(SCRIPT.values()))[:6])

## 5. Config

`RUN` picks which baseline(s) to train. `SAVE_EVERY` is the CRNN's
checkpoint cadence in optimiser steps — lower = less lost on a disconnect,
slightly more Drive I/O.

In [ ]:
RUN = 'both'          # 'kraken' | 'crnn' | 'both'
HEIGHT = 64           # line-image height; 48 is the other arm
MAX_EPOCHS = 60
BATCH = 32
LR = 3e-4
GRAD_CLIP = 5.0
PATIENCE = 10         # early stop on val CER
SAVE_EVERY = 200      # CRNN: checkpoint every N steps to Drive
EVAL_EVERY = 1        # epochs between val evals
SEED = 0
import random, numpy as np, torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 6. Kraken baseline

Warm-starts from the Muharaf recognition model, or from this run's own
latest checkpoint on Drive if one exists (disconnect resume). `--savefreq 1`
writes a model to Drive every epoch; `--lag {PATIENCE}` is Kraken's early stop.

In [ ]:
import glob, subprocess, pathlib, urllib.request
if RUN in ('kraken', 'both'):
    KOUT = WORK / 'kraken' / 'arams_ft'
    warm = sorted(glob.glob(str(WORK / 'kraken' / 'arams_ft_*.mlmodel')),
                  key=lambda p: pathlib.Path(p).stat().st_mtime)
    if warm:
        seed_model = warm[-1]; print('RESUME kraken from', seed_model)
    else:
        seed_model = str(WORK / 'kraken' / 'muharaf_rec.mlmodel')
        if not pathlib.Path(seed_model).exists():
            print('fetching Muharaf recognition model (Zenodo 14295489)...')
            rec = urllib.request.urlopen('https://zenodo.org/api/records/14295489').read()
            import json as _j
            url = _j.loads(rec)['files'][0]['links']['self']
            urllib.request.urlretrieve(url, seed_model)
        print('warm start from Muharaf model')
    train_list = [str(IMAGES / f'{i}.png') for i in SPLIT['train'] if (IMAGES / f'{i}.png').exists()]
    val_list   = [str(IMAGES / f'{i}.png') for i in SPLIT['val']   if (IMAGES / f'{i}.png').exists()]
    (WORK / 'kraken' / 'train.files').write_text('\n'.join(train_list))
    (WORK / 'kraken' / 'val.files').write_text('\n'.join(val_list))
    cmd = ['ketos', 'train', '-f', 'path', '-i', seed_model, '--resize', 'add',
           '-o', str(KOUT), '-d', 'cuda:0', '--savefreq', '1',
           '--min-epochs', '30', '--lag', str(PATIENCE), '--augment',
           '--workers', '2', '-t', str(WORK / 'kraken' / 'train.files'),
           '-e', str(WORK / 'kraken' / 'val.files')]
    print(' '.join(cmd))
    subprocess.run(cmd, check=False)
else:
    print('RUN != kraken — skipping')

## 6b. Kraken eval — weighted + per-script CER vs RefLAM

In [ ]:
import glob, subprocess, re, pathlib
if RUN in ('kraken', 'both'):
    best = sorted(glob.glob(str(WORK / 'kraken' / 'arams_ft_best.mlmodel')) or
                  glob.glob(str(WORK / 'kraken' / 'arams_ft_*.mlmodel')),
                  key=lambda p: pathlib.Path(p).stat().st_mtime)[-1]
    print('eval model:', best)
    def _ktest(ids):
        files = [str(IMAGES / f'{i}.png') for i in ids if (IMAGES / f'{i}.png').exists()]
        lf = WORK / 'kraken' / '_evalset.files'; lf.write_text('\n'.join(files))
        out = subprocess.run(['ketos', 'test', '-m', best, '-f', 'path',
                              '-t', str(lf), '-d', 'cuda:0'],
                             capture_output=True, text=True).stdout
        m = re.search(r'([Cc]haracter [Aa]ccuracy|Accuracy)[^0-9]*([0-9.]+)%', out)
        return (100 - float(m.group(2))) if m else None, out[-400:]
    kr = {}
    kr['weighted'], _ = _ktest(SPLIT['test'])
    for sc in sorted({SCRIPT.get(i, '') for i in SPLIT['test']} - {''}):
        kr[sc], _ = _ktest([i for i in SPLIT['test'] if SCRIPT.get(i) == sc])
    print('Kraken CER %:', {k: round(v, 2) if v is not None else None for k, v in kr.items()})
    print('RefLAM      : weighted 23.31 | maghrebi 32.71 | naskh 22.62 | ruqah 11.65')
    (WORK / 'kraken' / 'result.json').write_text(json.dumps(kr, indent=2))

## 7. Own CRNN — self-contained, checkpoint-every-N-steps

Charset is the sorted set of characters in the **training** transcriptions
(index 0 reserved for the CTC blank). Images are greyscale, height-normalised
to `HEIGHT` preserving aspect, batches are width-sorted and right-padded.

In [ ]:
import pathlib, json, math, time, random
import torch, torch.nn as nn, numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from arabic_ocr_bench.metrics import cer as _cer
from arabic_ocr_bench.normalize import normalize as _norm

CK = WORK / 'crnn' / 'ckpt_last.pt'
MET = WORK / 'crnn' / 'metrics.jsonl'

def _texts(ids):
    return {i: (IMAGES / f'{i}.gt.txt').read_text(encoding='utf-8').strip()
            for i in ids if (IMAGES / f'{i}.gt.txt').exists()}
TRAIN_T, VAL_T, TEST_T = _texts(SPLIT['train']), _texts(SPLIT['val']), _texts(SPLIT['test'])

if CK.exists():
    _st = torch.load(CK, map_location='cpu')
    CHARSET = _st['charset']; print('RESUME crnn @ step', _st['step'], '| charset', len(CHARSET))
else:
    CHARSET = ['<blank>'] + sorted({c for t in TRAIN_T.values() for c in t})
    _st = None; print('fresh crnn | charset', len(CHARSET))
STOI = {c: i for i, c in enumerate(CHARSET)}
OOV = sum(1 for t in list(TRAIN_T.values()) for c in t if c not in STOI)
print('train-charset OOV chars:', OOV)

class Lines(Dataset):
    def __init__(self, texts): self.items = sorted(texts.items())
    def __len__(self): return len(self.items)
    def __getitem__(self, k):
        sid, txt = self.items[k]
        im = Image.open(IMAGES / f'{sid}.png').convert('L')
        w = max(8, int(im.width * HEIGHT / im.height))
        im = im.resize((w, HEIGHT))
        x = torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0)[None]  # 1xHxW
        y = torch.tensor([STOI[c] for c in txt if c in STOI], dtype=torch.long)
        return x, y, txt

def collate(b):
    b = [x for x in b if len(x[1]) > 0]
    b.sort(key=lambda x: x[0].shape[2], reverse=True)
    xs, ys, ts = zip(*b)
    W = max(x.shape[2] for x in xs)
    X = torch.zeros(len(xs), 1, HEIGHT, W)
    for i, x in enumerate(xs): X[i, :, :, :x.shape[2]] = x
    xl = torch.tensor([x.shape[2] // 4 for x in xs], dtype=torch.long)  # /4 downsample
    yl = torch.tensor([len(y) for y in ys], dtype=torch.long)
    return X, torch.cat(ys), xl, yl, ts

class CRNN(nn.Module):
    def __init__(self, n_cls, h=HEIGHT):
        super().__init__()
        def blk(i, o, pool): return [nn.Conv2d(i, o, 3, 1, 1), nn.BatchNorm2d(o), nn.ReLU(True)] + ([nn.MaxPool2d(pool)] if pool else [])
        self.cnn = nn.Sequential(
            *blk(1, 64, (2, 2)), *blk(64, 128, (2, 2)),
            *blk(128, 256, None), *blk(256, 256, (2, 1)),
            *blk(256, 512, None), *blk(512, 512, (2, 1)),
            nn.Conv2d(512, 512, (h // 8, 1)), nn.ReLU(True))
        self.rnn = nn.LSTM(512, 256, 2, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(512, n_cls)
    def forward(self, x):
        f = self.cnn(x)                      # B x 512 x 1 x W/4
        f = f.squeeze(2).permute(0, 2, 1)     # B x W/4 x 512
        f, _ = self.rnn(f)
        return self.fc(f).log_softmax(2)      # B x T x n_cls

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
net = CRNN(len(CHARSET)).to(dev)
opt = torch.optim.AdamW(net.parameters(), lr=LR)
steps_per_epoch = max(1, len(TRAIN_T) // BATCH)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS * steps_per_epoch)
ctc = nn.CTCLoss(blank=0, zero_infinity=True)
start_step, best_cer, bad = 0, 9.9, 0
if _st is not None:
    net.load_state_dict(_st['model']); opt.load_state_dict(_st['opt'])
    sched.load_state_dict(_st['sched'])
    start_step, best_cer, bad = _st['step'], _st['best_cer'], _st['bad']

## 7b. Train loop — resumes from `start_step`, saves every `SAVE_EVERY` steps

In [ ]:
if RUN in ('crnn', 'both'):
    tl = DataLoader(Lines(TRAIN_T), BATCH, shuffle=True, num_workers=2, collate_fn=collate, drop_last=True)
    vl = DataLoader(Lines(VAL_T), BATCH, shuffle=False, num_workers=2, collate_fn=collate)

    def _save(step):
        tmp = CK.with_suffix('.tmp')
        torch.save({'model': net.state_dict(), 'opt': opt.state_dict(),
                    'sched': sched.state_dict(), 'step': step, 'best_cer': best_cer,
                    'bad': bad, 'charset': CHARSET}, tmp)
        tmp.replace(CK)

    @torch.no_grad()
    def _decode(logp):
        idx = logp.argmax(2).cpu().numpy()
        outs = []
        for row in idx:
            prev, s = 0, []
            for t in row:
                if t != prev and t != 0: s.append(CHARSET[t])
                prev = t
            outs.append(''.join(s))
        return outs

    @torch.no_grad()
    def _eval(loader, texts_by_script=False):
        net.eval(); num = {'raw': 0.0, 'light': 0.0, 'diac_insensitive': 0.0}; den = 0
        per = {}
        for X, _, xl, _, ts in loader:
            pred = _decode(net(X.to(dev)))
            for p, g in zip(pred, ts):
                for prof in num:
                    num[prof] += _cer(_norm(g, prof), _norm(p, prof))
                den += 1
        net.train()
        return {k: v / max(1, den) for k, v in num.items()}

    step = start_step
    ep0 = step // steps_per_epoch
    for epoch in range(ep0, MAX_EPOCHS):
        for X, Y, xl, yl, ts in tl:
            logp = net(X.to(dev)).permute(1, 0, 2)          # T x B x C
            loss = ctc(logp, Y.to(dev), xl.clamp(min=1), yl)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP)
            opt.step(); sched.step(); step += 1
            if step % SAVE_EVERY == 0:
                _save(step)
                print(f'  step {step}  loss {loss.item():.3f}  (checkpoint)', flush=True)
        if epoch % EVAL_EVERY == 0:
            m = _eval(vl)
            row = {'epoch': epoch, 'step': step, 'val_cer': m, 'ts': time.time()}
            with open(MET, 'a') as fh: fh.write(json.dumps(row) + '\n')
            print(f'epoch {epoch}  val CER L1/L2/L3 = '
                  f"{m['raw']:.4f}/{m['light']:.4f}/{m['diac_insensitive']:.4f}", flush=True)
            cur = m['diac_insensitive']
            if cur < best_cer - 1e-4:
                best_cer, bad = cur, 0
                torch.save({'model': net.state_dict(), 'charset': CHARSET, 'val_cer': m},
                           WORK / 'crnn' / 'best.pt')
            else:
                bad += 1
            _save(step)
            if bad >= PATIENCE:
                print('early stop — no val-CER improvement for', PATIENCE, 'evals'); break
    print('CRNN training done. best val CER(L3) =', round(best_cer, 4))

## 7c. CRNN test — CER at L1/L2/L3 + per-script

In [ ]:
if RUN in ('crnn', 'both'):
    bp = WORK / 'crnn' / 'best.pt'
    sd = torch.load(bp, map_location=dev) if bp.exists() else {'model': net.state_dict(), 'charset': CHARSET}
    net.load_state_dict(sd['model']); net.eval()
    from collections import defaultdict
    agg = defaultdict(lambda: {'raw': 0.0, 'light': 0.0, 'diac_insensitive': 0.0, 'n': 0})
    dl = DataLoader(Lines(TEST_T), BATCH, shuffle=False, num_workers=2, collate_fn=collate)
    # map each test text back to its script via the sorted Lines order
    ids_sorted = [i for i, _ in sorted(TEST_T.items())]
    it = iter(ids_sorted)
    with torch.no_grad():
        for X, _, xl, _, ts in dl:
            preds = []
            idx = net(X.to(dev)).argmax(2).cpu().numpy()
            for row in idx:
                prev, s = 0, []
                for t in row:
                    if t != prev and t != 0: s.append(CHARSET[t])
                    prev = t
                preds.append(''.join(s))
            for p, g in zip(preds, ts):
                sid = next(it)
                for grp in ('all', SCRIPT.get(sid, 'unknown')):
                    for prof in ('raw', 'light', 'diac_insensitive'):
                        agg[grp][prof] += _cer(_norm(g, prof), _norm(p, prof))
                    agg[grp]['n'] += 1
    res = {g: {p: round(v[p] / max(1, v['n']), 4) for p in ('raw','light','diac_insensitive')} | {'n': v['n']}
           for g, v in agg.items()}
    print(json.dumps(res, indent=2, ensure_ascii=False))
    print('RefLAM Kraken: weighted 23.31 | maghrebi 32.71 | naskh 22.62 | ruqah 11.65')
    (WORK / 'crnn' / 'result.json').write_text(json.dumps(res, indent=2, ensure_ascii=False))

## 8. Commit the results JSON back to the repo

Only the small `result.json` files — never the checkpoints.

In [ ]:
import shutil, subprocess, time, pathlib
rid = time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())
dst = pathlib.Path('results') / f'ctc-baseline-{rid}'
dst.mkdir(parents=True, exist_ok=True)
for p in (WORK / 'kraken' / 'result.json', WORK / 'crnn' / 'result.json', WORK / 'crnn' / 'metrics.jsonl'):
    if p.exists(): shutil.copy2(p, dst / p.name)
if os.environ.get('GH_TOKEN'):
    subprocess.run(['git', 'config', 'user.email', 'colab@arabic-ocr-bench'])
    subprocess.run(['git', 'config', 'user.name', 'colab:ctc-baseline'])
    subprocess.run(['git', 'add', '-f', str(dst)])
    subprocess.run(['git', 'commit', '-m', f'Tier-0 0.1 CTC baseline results ({rid})'])
    subprocess.run(['git', 'pull', '--rebase', '--autostash', 'origin', 'main'])
    print(subprocess.run(['git', 'push', 'origin', 'HEAD:main'], capture_output=True, text=True).stderr[-300:])
else:
    print('no GH_TOKEN — download', dst, 'from the file browser')

## Resume / notes

- **Disconnected?** Reopen, `Runtime → Run all`. Cell 6 warm-starts Kraken from
  its newest `arams_ft_*.mlmodel` on Drive; cell 7 loads `crnn/ckpt_last.pt`
  and continues from that step. Nothing before the last `SAVE_EVERY` step (or
  the last epoch, for Kraken) is lost.
- **`metrics.jsonl`** on Drive is the per-epoch val-CER trace — plot it or
  recover the best epoch even if the notebook never reached the final cell.
- If the CRNN CER is far from the RefLAM anchors, check: charset size / OOV
  print in cell 7, the `HEIGHT` normalisation, and that `xl` (the CTC input
  lengths) matches the CNN's width downsampling factor (here /4).
- The released AraMS `.gt.txt` is already the *normalised* transcription, so
  L1≈L2≈L3 here will be closer than on raw-diacritised corpora.